# Narrative Shift Detection Pipeline
# Data Preprocessing Notebook

---

## Pipeline Overview

**Stage 0:** Raw Dataset - Define paths and libraries  
**Stage 1:** Sentence Segmentation - Break articles into sentences  
**Stage 2:** Context-Aware Sentence Embedding - Encode sentences with neighbors using SBERT  
**Stage 3:** Topic Embedding Construction - Create semantic topic representations  
**Stage 4:** Sentence-Level Topic Weighting - Compute cosine similarity to topics  
**Stage 5:** Topic Filtering and Temporal Ordering - Filter by threshold & sort by date  
**Stage 6:** Same-Date Aggregation - Aggregate same-day articles via mean pooling  
**Stage 7:** Topic-Wise Adaptive Temporal Windowing - Create overlapping temporal windows  
**Stage 8:** Temporal Contrastive Learning (TCL) - Learn shift-sensitive representations  
**Stage 9:** Narrative Shift Scoring - Compute shift magnitude and detect changes  

---

### Key Pipeline Characteristics:
- **Unsupervised:** No labeled data required
- **Entity-Agnostic:** Detects semantic shifts, not entity changes
- **Multi-Topic:** Handles multiple topics simultaneously
- **Fine-Grained:** Operates at sentence level, aggregates temporally
- **Scalable:** Designed for large-scale news corpora

---

---

## Stage 0: Raw Dataset

**Input:** A raw corpus of news articles represented as a table with two columns:
- `Date`: publication date
- `Article`: full article content

**Example Input:**
```
Date: 2022-06-15
Article: "Artificial intelligence is rapidly expanding. Data centers now consume massive 
amounts of electricity. Governments are beginning to regulate AI usage."
```

**Motivation:** 
Articles are long, may contain multiple topics, and often mix several narrative frames. 
Narrative shifts occur at a finer granularity than the article level, motivating further 
decomposition.

**Stage 0 Goal:** 
- Define all data paths available in Data_Files folder
- Import necessary libraries
- Set up configuration (NO data loading yet)

---

In [1]:
# ============================================================================
# STAGE 0: IMPORT ALL REQUIRED LIBRARIES
# ============================================================================
# Run this cell first if you want to run Stage 0 independently
# ============================================================================

print("=" * 100)
print("STAGE 0: IMPORTING REQUIRED LIBRARIES")
print("=" * 100)

# Core data manipulation
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Date and time handling
from datetime import datetime, timedelta

# File operations
import json
import csv
import glob

# Display and progress
from IPython.display import display, clear_output
from tqdm import tqdm

print("✅ All Stage 0 libraries imported successfully")
print("\n" + "=" * 100)
print("✅ STAGE 0 LIBRARIES READY - You can now run Stage 0 cells")
print("=" * 100)

STAGE 0: IMPORTING REQUIRED LIBRARIES
✅ All Stage 0 libraries imported successfully

✅ STAGE 0 LIBRARIES READY - You can now run Stage 0 cells


In [9]:
# ============================================================================
# STAGE 0: RAW DATASET - SETUP AND PATH DEFINITION
# ============================================================================
# Purpose: Define all data paths and import necessary libraries
# Note: NO data loading in this stage - only setup and path configuration
# ============================================================================

print("=" * 100)
print("STAGE 0: RAW DATASET - SETUP AND PATH DEFINITION")
print("=" * 100)

# ----------------------------------------------------------------------------
# 1. Import Required Libraries
# ----------------------------------------------------------------------------
print("\n📦 Importing Libraries...")

# Core data manipulation
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Date and time handling
from datetime import datetime, timedelta

# File operations
import json
import csv
import glob

# Display and progress
from IPython.display import display, clear_output
from tqdm import tqdm

print("✅ Core libraries imported successfully")

# ----------------------------------------------------------------------------
# 2. Define Data Folder Path
# ----------------------------------------------------------------------------
print("\n📁 Defining Data Folder Path...")

# Main data folder containing all CSV files
DATA_FILES_FOLDER = Path('/home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_0')

# Verify folder exists
if DATA_FILES_FOLDER.exists():
    print(f"✅ Data folder found: {DATA_FILES_FOLDER.absolute()}")
else:
    print(f"❌ Data folder not found: {DATA_FILES_FOLDER.absolute()}")
    print("   Please ensure the Data_Files folder exists in the same directory as this notebook")

# ----------------------------------------------------------------------------
# 3. Discover and Define All Data File Paths
# ----------------------------------------------------------------------------
print("\n🔍 Discovering Data Files...")

# Get all CSV files in the Data_Files folder
data_file_paths = sorted(DATA_FILES_FOLDER.glob('Data_*.csv'))

# Extract file numbers for sorting (Data_1.csv -> 1, Data_10.csv -> 10, etc.)
def extract_file_number(file_path):
    """Extract numeric part from filename like 'Data_123.csv' -> 123"""
    try:
        return int(file_path.stem.split('_')[1])
    except:
        return 0

# Sort files by number (Data_1, Data_2, ..., Data_347)
data_file_paths = sorted(data_file_paths, key=extract_file_number)

# Convert to list of Path objects
DATA_FILE_PATHS = data_file_paths

print(f"✅ Discovered {len(DATA_FILE_PATHS)} data files")

# ----------------------------------------------------------------------------
# 4. Display File Path Information
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("📊 DATA FILE INVENTORY")
print("=" * 100)

print(f"\n📈 Total Files: {len(DATA_FILE_PATHS)}")
print(f"📂 Location: {DATA_FILES_FOLDER.absolute()}")

# Show first 10 and last 10 files
print(f"\n📋 First 10 Files:")
for i, file_path in enumerate(DATA_FILE_PATHS[:10], 1):
    print(f"   {i:3d}. {file_path.name}")

if len(DATA_FILE_PATHS) > 20:
    print(f"\n   ... ({len(DATA_FILE_PATHS) - 20} files omitted) ...")

if len(DATA_FILE_PATHS) > 10:
    print(f"\n📋 Last 10 Files:")
    for i, file_path in enumerate(DATA_FILE_PATHS[-10:], len(DATA_FILE_PATHS) - 9):
        print(f"   {i:3d}. {file_path.name}")

# ----------------------------------------------------------------------------
# 5. Configuration Settings
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("⚙️  CONFIGURATION SETTINGS")
print("=" * 100)

# Expected columns in each CSV file
EXPECTED_COLUMNS = ['Date', 'Article', 'Source']

# Data configuration
CONFIG = {
    'data_folder': str(DATA_FILES_FOLDER),
    'total_files': len(DATA_FILE_PATHS),
    'expected_columns': EXPECTED_COLUMNS,
    'articles_per_file': 10000,  # Expected number of articles per file
    'date_column': 'Date',
    'article_column': 'Article',
    'source_column': 'Source'
}

print(f"\n📊 Configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

# ----------------------------------------------------------------------------
# 6. Create Helper Functions for File Access
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("🔧 HELPER FUNCTIONS DEFINED")
print("=" * 100)

def get_file_path(file_number):
    """
    Get the path for a specific data file by number.
    
    Args:
        file_number (int): File number (1 to total_files)
    
    Returns:
        Path: File path or None if not found
    """
    if 1 <= file_number <= len(DATA_FILE_PATHS):
        return DATA_FILE_PATHS[file_number - 1]
    else:
        print(f"⚠️  Invalid file number: {file_number} (valid range: 1-{len(DATA_FILE_PATHS)})")
        return None

def get_all_file_paths():
    """
    Get all data file paths.
    
    Returns:
        list: List of all file paths
    """
    return DATA_FILE_PATHS

def get_file_count():
    """
    Get total number of data files.
    
    Returns:
        int: Number of files
    """
    return len(DATA_FILE_PATHS)

def estimate_total_articles():
    """
    Estimate total number of articles across all files.
    
    Returns:
        int: Estimated total articles
    """
    return len(DATA_FILE_PATHS) * CONFIG['articles_per_file']

print("✅ Helper functions defined:")
print("   - get_file_path(file_number): Get path for specific file")
print("   - get_all_file_paths(): Get all file paths")
print("   - get_file_count(): Get total number of files")
print("   - estimate_total_articles(): Estimate total articles")

# ----------------------------------------------------------------------------
# 7. Quick Statistics (without loading data)
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("📈 ESTIMATED STATISTICS")
print("=" * 100)

estimated_articles = estimate_total_articles()

print(f"\n📊 Raw Dataset Overview (estimated):")
print(f"   Total Files: {get_file_count():,}")
print(f"   Estimated Articles: ~{estimated_articles:,}")
print(f"   Articles per File: ~{CONFIG['articles_per_file']:,}")
print(f"   Expected Columns: {', '.join(CONFIG['expected_columns'])}")

# ----------------------------------------------------------------------------
# 8. Path Export for Next Stages
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("✅ STAGE 0 COMPLETE - PATHS AND CONFIGURATION READY")
print("=" * 100)

print(f"\n💡 Available Variables:")
print(f"   - DATA_FILE_PATHS: List of {len(DATA_FILE_PATHS)} file paths")
print(f"   - DATA_FILES_FOLDER: Path object pointing to Data_Files folder")
print(f"   - CONFIG: Configuration dictionary")
print(f"   - Helper functions: get_file_path(), get_all_file_paths(), etc.")

print(f"\n📌 Next Stage: Load and process articles from these files")
print(f"=" * 100)

STAGE 0: RAW DATASET - SETUP AND PATH DEFINITION

📦 Importing Libraries...
✅ Core libraries imported successfully

📁 Defining Data Folder Path...
✅ Data folder found: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_0

🔍 Discovering Data Files...
✅ Discovered 113 data files

📊 DATA FILE INVENTORY

📈 Total Files: 113
📂 Location: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_0

📋 First 10 Files:
     1. Data_1.csv
     2. Data_2.csv
     3. Data_3.csv
     4. Data_4.csv
     5. Data_5.csv
     6. Data_6.csv
     7. Data_7.csv
     8. Data_8.csv
     9. Data_9.csv
    10. Data_10.csv

   ... (93 files omitted) ...

📋 Last 10 Files:
   104. Data_104.csv
   105. Data_105.csv
   106. Data_106.csv
   107. Data_107.csv
   108. Data_108.csv
   109. Data_109.csv
   110. Data_110.csv
   111. Data_111.csv
   112. Data_112.csv
   113. Data_113.csv

⚙️  CONFIGURATION SETTINGS

📊 Configuration:
   data_folder: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_0
   total_files: 113
  

---

## Stage 1: Sentence Segmentation

**Goal:** Break each article into sentences while preserving temporal order and context

**Input:** Raw articles from Data_Files (Date, Article, Source)

**Output Format:**
- `sentence_id`: Unique identifier for each sentence
- `article_id`: Original article identifier
- `date`: Publication date (preserved from article)
- `source`: Article source
- `previous_sentence_1`: The sentence 2 positions before (or "" if not available)
- `previous_sentence_2`: The sentence 1 position before (or "" if first)
- `main_sentence`: Current sentence
- `next_sentence_1`: The sentence 1 position after (or "" if last)
- `next_sentence_2`: The sentence 2 positions after (or "" if not available)

**Context Window:** Size 5 (2 previous + 1 main + 2 next sentences)

**Why this structure:**
- Preserves sentence order within articles
- Provides wider context (2 sentences before and after) for embedding in Stage 2
- Enables richer context-aware semantic encoding
- Window size of 5 captures more surrounding narrative context

**Processing Strategy:**
- Multi-threaded file processing (one file at a time)
- Each file processed completely before moving to next
- Output saved to `Processed_Data/Stage_1/` folder

---

In [23]:
# ============================================================================
# STAGE 1: IMPORT ALL REQUIRED LIBRARIES
# ============================================================================
# Run this cell first if you want to run Stage 1 independently
# ============================================================================

print("=" * 100)
print("STAGE 1: IMPORTING REQUIRED LIBRARIES")
print("=" * 100)

# Core libraries from Stage 0
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
import json
import csv
import glob
from IPython.display import display, clear_output
from tqdm import tqdm

# Stage 1 specific libraries
import nltk
from nltk.tokenize import sent_tokenize
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

print("✅ All libraries imported successfully")

# Define paths if not already defined
if 'DATA_FILES_FOLDER' not in dir():
    DATA_FILES_FOLDER = Path('Data_Files')
    print(f"✅ DATA_FILES_FOLDER defined: {DATA_FILES_FOLDER.absolute()}")

if 'STAGE_1_OUTPUT_FOLDER' not in dir():
    STAGE_1_OUTPUT_FOLDER = Path('Processed_Data/Stage_1')
    print(f"✅ STAGE_1_OUTPUT_FOLDER defined: {STAGE_1_OUTPUT_FOLDER.absolute()}")

# Helper functions if not already defined
if 'get_all_file_paths' not in dir():
    def get_all_file_paths():
        """Get all data file paths"""
        data_file_paths = sorted(DATA_FILES_FOLDER.glob('Data_*.csv'))
        def extract_file_number(file_path):
            try:
                return int(file_path.stem.split('_')[1])
            except:
                return 0
        return sorted(data_file_paths, key=extract_file_number)
    print("✅ Helper function get_all_file_paths() defined")

print("\n" + "=" * 100)
print("✅ STAGE 1 LIBRARIES READY - You can now run Stage 1 cells")
print("=" * 100)

STAGE 1: IMPORTING REQUIRED LIBRARIES
✅ All libraries imported successfully

✅ STAGE 1 LIBRARIES READY - You can now run Stage 1 cells


In [24]:
# ============================================================================
# STAGE 1: SENTENCE SEGMENTATION - SETUP
# ============================================================================
# Import NLP library for sentence tokenization
# ============================================================================

print("=" * 100)
print("STAGE 1: SENTENCE SEGMENTATION - SETUP")
print("=" * 100)

# Import sentence tokenization library
try:
    import nltk
    from nltk.tokenize import sent_tokenize
    print("✅ NLTK already installed")
except ImportError:
    print("📥 Installing NLTK...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk'])
    import nltk
    from nltk.tokenize import sent_tokenize
    print("✅ NLTK installed successfully")

# Download punkt tokenizer if not already downloaded
try:
    nltk.data.find('tokenizers/punkt')
    print("✅ Punkt tokenizer already available")
except LookupError:
    print("📥 Downloading punkt tokenizer...")
    nltk.download('punkt', quiet=True)
    print("✅ Punkt tokenizer downloaded")

# Import threading libraries
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

print("\n✅ All libraries for Stage 1 ready")

# ----------------------------------------------------------------------------
# Configure Output Folder
# ----------------------------------------------------------------------------
print("\n📁 Configuring Output Folder...")

# Create output folder for Stage 1
STAGE_1_OUTPUT_FOLDER = Path('/home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_1')
STAGE_1_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"✅ Output folder: {STAGE_1_OUTPUT_FOLDER.absolute()}")

# Configuration for Stage 1
STAGE_1_CONFIG = {
    'output_folder': str(STAGE_1_OUTPUT_FOLDER),
    'output_file_prefix': 'Data_s1_',
    'columns': ['sentence_id', 'article_id', 'date', 'source', 
                'previous_sentence_1', 'previous_sentence_2', 'main_sentence', 
                'next_sentence_1', 'next_sentence_2'],
    'window_size': 5,  # Context window: 2 previous + 1 main + 2 next
    'num_threads': 8,  # Number of parallel threads for file processing
    'chunk_size': 1000  # Process articles in chunks for memory efficiency
}

print(f"\n⚙️  Stage 1 Configuration:")
for key, value in STAGE_1_CONFIG.items():
    print(f"   {key}: {value}")

print("\n" + "=" * 100)
print("✅ STAGE 1 SETUP COMPLETE")
print("=" * 100)

STAGE 1: SENTENCE SEGMENTATION - SETUP
✅ NLTK already installed
✅ Punkt tokenizer already available

✅ All libraries for Stage 1 ready

📁 Configuring Output Folder...
✅ Output folder: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_1

⚙️  Stage 1 Configuration:
   output_folder: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_1
   output_file_prefix: Data_s1_
   columns: ['sentence_id', 'article_id', 'date', 'source', 'previous_sentence_1', 'previous_sentence_2', 'main_sentence', 'next_sentence_1', 'next_sentence_2']
   window_size: 5
   num_threads: 8
   chunk_size: 1000

✅ STAGE 1 SETUP COMPLETE


In [25]:
# ============================================================================
# STAGE 1: SENTENCE SEGMENTATION - PROCESSING FUNCTIONS
# ============================================================================
# Define functions for sentence segmentation with context
# ============================================================================

def segment_article_into_sentences(article_text, article_id, date, source, window_size=5):
    """
    Segment a single article into sentences with context window (2 previous + main + 2 next).
    
    Args:
        article_text (str): Full article text
        article_id (str): Unique article identifier
        date (str): Publication date
        source (str): Article source
        window_size (int): Total context window size (default: 5 = 2 previous + 1 main + 2 next)
    
    Returns:
        list: List of sentence dictionaries with context
    """
    # Tokenize article into sentences
    sentences = sent_tokenize(str(article_text))
    
    # Calculate context size (sentences before and after main sentence)
    context_before = (window_size - 1) // 2  # 2 sentences before
    context_after = window_size - 1 - context_before  # 2 sentences after
    
    # Create sentence records with context
    sentence_records = []
    
    for i, main_sentence in enumerate(sentences):
        # Skip empty sentences
        if not main_sentence or str(main_sentence).strip() == "":
            continue
        
        # Get 2 previous sentences (or fewer if at beginning)
        previous_sentence_1 = sentences[i - 2] if i >= 2 else ""
        previous_sentence_2 = sentences[i - 1] if i >= 1 else ""
        
        # Get 2 next sentences (or fewer if at end)
        next_sentence_1 = sentences[i + 1] if i < len(sentences) - 1 else ""
        next_sentence_2 = sentences[i + 2] if i < len(sentences) - 2 else ""
        
        # Create unique sentence ID (use original index i+1 to maintain sequence)
        sentence_id = f"{article_id}_s{len(sentence_records) + 1}"
        
        # Create sentence record - ensure empty strings instead of None/NaN
        record = {
            'sentence_id': sentence_id,
            'article_id': article_id,
            'date': date,
            'source': source,
            'previous_sentence_1': previous_sentence_1 if previous_sentence_1 else "",
            'previous_sentence_2': previous_sentence_2 if previous_sentence_2 else "",
            'main_sentence': main_sentence.strip(),
            'next_sentence_1': next_sentence_1 if next_sentence_1 else "",
            'next_sentence_2': next_sentence_2 if next_sentence_2 else ""
        }
        
        sentence_records.append(record)
    
    return sentence_records


def process_single_file(file_path, file_number, total_files):
    """
    Process a single data file and perform sentence segmentation.
    
    Args:
        file_path (Path): Path to the CSV file
        file_number (int): File number (for tracking)
        total_files (int): Total number of files
    
    Returns:
        tuple: (file_number, sentence_count, output_file_path, success)
    """
    try:
        start_time = time.time()
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Validate required columns
        required_cols = ['Date', 'Article', 'Source']
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️  Skipping {file_path.name}: Missing required columns")
            return (file_number, 0, None, False, 0)
        
        # Process each article
        all_sentences = []
        
        for idx, row in df.iterrows():
            article_id = f"f{file_number}_a{idx + 1}"
            date = row['Date']
            article_text = row['Article']
            source = row['Source']
            
            # Skip if article text is NaN or empty
            if pd.isna(article_text) or str(article_text).strip() == "":
                continue
            
            # Segment article into sentences
            sentences = segment_article_into_sentences(
                article_text, article_id, date, source
            )
            
            all_sentences.extend(sentences)
        
        # Create DataFrame from sentences
        sentences_df = pd.DataFrame(all_sentences)
        
        # Replace any remaining NaN values with empty strings
        sentences_df = sentences_df.fillna("")
        
        # Define output file path
        output_file = STAGE_1_OUTPUT_FOLDER / f"Data_s1_{file_number}.csv"
        
        # Save to CSV
        sentences_df.to_csv(output_file, index=False)
        
        elapsed_time = time.time() - start_time
        
        return (file_number, len(sentences_df), output_file, True, elapsed_time)
        
    except Exception as e:
        print(f"\n❌ Error processing {file_path.name}: {e}")
        return (file_number, 0, None, False, 0)


print("✅ Sentence segmentation functions defined:")
print("   - segment_article_into_sentences(): Segment one article")
print("   - process_single_file(): Process entire file")

✅ Sentence segmentation functions defined:
   - segment_article_into_sentences(): Segment one article
   - process_single_file(): Process entire file


In [26]:
# ============================================================================
# STAGE 1: SENTENCE SEGMENTATION - MAIN PROCESSING
# ============================================================================
# Process all files with multi-threading for improved speed
# ============================================================================

print("=" * 100)
print("STAGE 1: SENTENCE SEGMENTATION - MAIN PROCESSING")
print("=" * 100)

# Get all file paths
all_files = get_all_file_paths()
total_files = len(all_files)
num_threads = STAGE_1_CONFIG['num_threads']

print(f"\n📊 Processing Overview:")
print(f"   Total files to process: {total_files:,}")
print(f"   Output folder: {STAGE_1_OUTPUT_FOLDER.absolute()}")
print(f"   Number of threads: {num_threads}")
print(f"   Processing mode: Multi-threaded (parallel processing)")

# Initialize tracking variables
processed_files = 0
total_sentences = 0
successful_files = []
failed_files = []
start_time_overall = time.time()

# Thread tracking - stores currently processing files per thread
thread_status = {}  # {thread_id: {'file_number': X, 'file_name': 'Data_X.csv', 'start_time': time}}
completed_files = []  # Track recently completed files

# Thread-safe lock for updating progress
progress_lock = threading.Lock()

def mark_thread_start(thread_id, file_number, file_name):
    """Mark that a thread has started processing a file"""
    with progress_lock:
        thread_status[thread_id] = {
            'file_number': file_number,
            'file_name': file_name,
            'start_time': time.time()
        }

def mark_thread_complete(thread_id):
    """Mark that a thread has completed processing"""
    with progress_lock:
        if thread_id in thread_status:
            del thread_status[thread_id]

def update_progress(file_number, file_name, sentence_count, output_file, success, elapsed):
    """Thread-safe progress update"""
    global processed_files, total_sentences
    
    with progress_lock:
        if success:
            processed_files += 1
            total_sentences += sentence_count
            file_info = {
                'file_number': file_number,
                'file_name': file_name,
                'sentences': sentence_count,
                'output': output_file.name,
                'time': elapsed
            }
            successful_files.append(file_info)
            completed_files.append(file_info)
            
            # Keep only last 10 completed files for display
            if len(completed_files) > 10:
                completed_files.pop(0)
            
            # Calculate statistics
            avg_time_per_file = (time.time() - start_time_overall) / processed_files
            estimated_remaining_time = avg_time_per_file * (total_files - processed_files)
            
            # Clear and display progress
            clear_output(wait=True)
            print("=" * 120)
            print(f"🚀 PROCESSING IN PROGRESS - {num_threads} THREADS ACTIVE")
            print("=" * 120)
            
            # Overall Progress bar
            progress_pct = (processed_files / total_files) * 100
            bar_length = 60
            filled_length = int(bar_length * processed_files / total_files)
            bar = '█' * filled_length + '░' * (bar_length - filled_length)
            
            print(f"\n📊 Overall Progress:")
            print(f"   [{bar}] {progress_pct:.1f}%")
            print(f"   Files: {processed_files}/{total_files} | Sentences: {total_sentences:,}")
            print(f"   Time Elapsed: {(time.time() - start_time_overall)/60:.1f} min | ETA: {estimated_remaining_time/60:.1f} min")
            print(f"   Avg Speed: {avg_time_per_file:.2f}s/file | Processing Rate: {total_files/(time.time() - start_time_overall)*60:.1f} files/min")
            
            # Thread-by-thread status
            print(f"\n🔄 Active Threads ({len(thread_status)}/{num_threads}):")
            print("-" * 120)
            if thread_status:
                # Sort threads by thread ID for consistent display
                sorted_threads = sorted(thread_status.items(), key=lambda x: x[0])
                for thread_id, status in sorted_threads:
                    elapsed_thread = time.time() - status['start_time']
                    # Create mini progress bar for thread
                    thread_bar_length = 15
                    # Estimate progress based on average file time
                    if processed_files > 0:
                        thread_progress = min(1.0, elapsed_thread / avg_time_per_file)
                    else:
                        thread_progress = 0.3  # Default for first files
                    thread_filled = int(thread_bar_length * thread_progress)
                    thread_bar = '█' * thread_filled + '░' * (thread_bar_length - thread_filled)
                    
                    print(f"   Thread {thread_id:2d}: [{thread_bar}] Processing File {status['file_number']:3d} "
                          f"({status['file_name']:<20}) - {elapsed_thread:.1f}s")
            else:
                print("   All threads idle")
            
            # Show recently completed files
            print(f"\n✅ Recently Completed Files:")
            print("-" * 120)
            for file_info in completed_files[-8:]:  # Show last 8
                print(f"   File {file_info['file_number']:3d}: {file_info['output']:<25} | "
                      f"Sentences: {file_info['sentences']:>7,} | Time: {file_info['time']:>5.2f}s")
            
        else:
            failed_files.append({
                'file_number': file_number,
                'file_name': file_name
            })
            print(f"\n❌ Failed: File {file_number} - {file_name}")

print(f"\n🚀 Starting multi-threaded sentence segmentation...")
print("=" * 120)

# Initial progress display
clear_output(wait=True)
print("=" * 120)
print(f"🚀 STARTING PROCESSING - {num_threads} THREADS INITIALIZING")
print("=" * 120)
print(f"\n📊 Ready to process {total_files:,} files")
print(f"   Output: {STAGE_1_OUTPUT_FOLDER.absolute()}")

# Wrapper function to track thread activity
def process_file_with_tracking(file_path, file_idx, total_files, thread_id):
    """Wrapper to track thread activity during processing"""
    # Mark thread as started BEFORE processing
    mark_thread_start(thread_id, file_idx, file_path.name)
    
    try:
        # Process the file
        result = process_single_file(file_path, file_idx, total_files)
        return result
    finally:
        # Mark thread as complete AFTER processing (but this happens fast)
        mark_thread_complete(thread_id)

# Process files using ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    # Submit files in batches to control thread assignment
    futures = []
    future_to_info = {}
    
    for file_idx, file_path in enumerate(all_files, 1):
        thread_id = (file_idx - 1) % num_threads  # Round-robin thread assignment
        future = executor.submit(process_file_with_tracking, file_path, file_idx, total_files, thread_id)
        futures.append(future)
        future_to_info[future] = (file_idx, file_path, thread_id)
    
    # Process completed tasks as they finish
    for future in as_completed(futures):
        file_idx, file_path, thread_id = future_to_info[future]
        try:
            result = future.result()
            file_number, sentence_count, output_file, success, elapsed = result
            update_progress(file_number, file_path.name, sentence_count, output_file, success, elapsed)
        except Exception as e:
            mark_thread_complete(thread_id)  # Ensure thread is marked complete on error
            print(f"\n❌ Exception processing file {file_idx}: {e}")
            update_progress(file_idx, file_path.name, 0, None, False, 0)

# Calculate final statistics
elapsed_time_overall = time.time() - start_time_overall

# Clear and show final summary
clear_output(wait=True)

print("\n" + "=" * 100)
print("📈 STAGE 1 PROCESSING COMPLETE")
print("=" * 100)

print(f"\n✅ Summary:")
print(f"   Files processed successfully: {len(successful_files):,}/{total_files:,}")
print(f"   Files failed: {len(failed_files):,}")
print(f"   Total sentences generated: {total_sentences:,}")
print(f"   Total processing time: {elapsed_time_overall/60:.2f} minutes")
if len(successful_files) > 0:
    print(f"   Average time per file: {elapsed_time_overall/len(successful_files):.2f} seconds")
    print(f"   Average sentences per file: {total_sentences/len(successful_files):.0f}")
print(f"   Processing speed: {total_files/(elapsed_time_overall/60):.1f} files/minute")

# Show sample of successful files
if successful_files:
    print(f"\n📋 Sample of Processed Files (first 10):")
    print("-" * 100)
    print(f"{'File #':<10} {'Sentences':<12} {'Output File':<30} {'Time (s)':<10}")
    print("-" * 100)
    # Sort by file number for display
    sorted_files = sorted(successful_files, key=lambda x: x['file_number'])
    for file_info in sorted_files[:10]:
        print(f"{file_info['file_number']:<10} {file_info['sentences']:<12,} "
              f"{file_info['output']:<30} {file_info['time']:<10.2f}")
    
    if len(successful_files) > 10:
        print(f"\n   ... ({len(successful_files) - 10} more files)")

# Show failed files if any
if failed_files:
    print(f"\n⚠️  Failed Files ({len(failed_files)}):")
    for file_info in sorted(failed_files, key=lambda x: x['file_number']):
        print(f"   {file_info['file_number']}. {file_info['file_name']}")

# Save processing summary
summary_file = STAGE_1_OUTPUT_FOLDER / 'processing_summary.json'
summary_data = {
    'total_files': total_files,
    'successful_files': len(successful_files),
    'failed_files': len(failed_files),
    'total_sentences': total_sentences,
    'processing_time_minutes': elapsed_time_overall / 60,
    'average_time_per_file_seconds': elapsed_time_overall / len(successful_files) if len(successful_files) > 0 else 0,
    'processing_speed_files_per_minute': total_files / (elapsed_time_overall / 60),
    'num_threads': num_threads,
    'output_folder': str(STAGE_1_OUTPUT_FOLDER),
    'timestamp': datetime.now().isoformat()
}

with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\n💾 Processing summary saved: {summary_file.name}")

print("\n" + "=" * 100)
print("🎉 STAGE 1 COMPLETE!")
print("=" * 100)
print(f"\n💡 Next Stage: Context-Aware Sentence Embedding (Stage 2)")
print(f"   Input: {len(successful_files):,} files with {total_sentences:,} sentences")
print(f"   Location: {STAGE_1_OUTPUT_FOLDER.absolute()}")
print("=" * 100)


📈 STAGE 1 PROCESSING COMPLETE

✅ Summary:
   Files processed successfully: 113/113
   Files failed: 0
   Total sentences generated: 3,237,195
   Total processing time: 2.78 minutes
   Average time per file: 1.47 seconds
   Average sentences per file: 28648
   Processing speed: 40.7 files/minute

📋 Sample of Processed Files (first 10):
----------------------------------------------------------------------------------------------------
File #     Sentences    Output File                    Time (s)  
----------------------------------------------------------------------------------------------------
1          14,704       Data_s1_1.csv                  7.10      
2          16,416       Data_s1_2.csv                  8.14      
3          23,443       Data_s1_3.csv                  10.83     
4          22,924       Data_s1_4.csv                  13.24     
5          24,972       Data_s1_5.csv                  13.32     
6          24,629       Data_s1_6.csv                  12.19    

In [28]:
# ============================================================================
# STAGE 1: VERIFICATION AND SAMPLE DISPLAY
# ============================================================================
# Verify output and display sample sentences
# ============================================================================

print("=" * 100)
print("STAGE 1: VERIFICATION - SAMPLE OUTPUT")
print("=" * 100)

# Get list of output files
output_files = sorted(STAGE_1_OUTPUT_FOLDER.glob('Data_s1_*.csv'))

if len(output_files) > 0:
    print(f"\n✅ Found {len(output_files):,} output files")
    
    # Load first file as sample
    sample_file = output_files[0]
    sample_df = pd.read_csv(sample_file)
    
    print(f"\n📊 Sample Data from: {sample_file.name}")
    print(f"   Total sentences in file: {len(sample_df):,}")
    print(f"   Columns: {list(sample_df.columns)}")
    
    # Display first 5 sentences
    print(f"\n📋 First 5 Sentences (showing window structure):")
    print("=" * 100)
    
    for idx in range(min(5, len(sample_df))):
        row = sample_df.iloc[idx]
        
        # Handle NaN values - convert to string safely (window size 5)
        prev_sent_1 = str(row['previous_sentence_1']) if pd.notna(row['previous_sentence_1']) else ""
        prev_sent_2 = str(row['previous_sentence_2']) if pd.notna(row['previous_sentence_2']) else ""
        main_sent = str(row['main_sentence']) if pd.notna(row['main_sentence']) else ""
        next_sent_1 = str(row['next_sentence_1']) if pd.notna(row['next_sentence_1']) else ""
        next_sent_2 = str(row['next_sentence_2']) if pd.notna(row['next_sentence_2']) else ""
        
        print(f"\n🔹 Sentence {idx + 1}:")
        print(f"   Sentence ID: {row['sentence_id']}")
        print(f"   Article ID: {row['article_id']}")
        print(f"   Date: {row['date']}")
        print(f"   Source: {row['source']}")
        print(f"   Previous-1 (i-2): \"{prev_sent_1[:50]}{'...' if len(prev_sent_1) > 50 else ''}\"")
        print(f"   Previous-2 (i-1): \"{prev_sent_2[:50]}{'...' if len(prev_sent_2) > 50 else ''}\"")
        print(f"   Main (i): \"{main_sent[:60]}{'...' if len(main_sent) > 60 else ''}\"")
        print(f"   Next-1 (i+1): \"{next_sent_1[:50]}{'...' if len(next_sent_1) > 50 else ''}\"")
        print(f"   Next-2 (i+2): \"{next_sent_2[:50]}{'...' if len(next_sent_2) > 50 else ''}\"")
    
    # Statistics
    print(f"\n\n📊 Statistics for Sample File:")
    print("-" * 100)
    
    # Count unique articles
    unique_articles = sample_df['article_id'].nunique()
    avg_sentences_per_article = len(sample_df) / unique_articles
    
    print(f"   Unique articles: {unique_articles:,}")
    print(f"   Average sentences per article: {avg_sentences_per_article:.1f}")
    
    # Count sentences with/without context (handle NaN) - window size 5
    has_prev_1 = sample_df['previous_sentence_1'].notna() & (sample_df['previous_sentence_1'] != "")
    has_prev_2 = sample_df['previous_sentence_2'].notna() & (sample_df['previous_sentence_2'] != "")
    has_next_1 = sample_df['next_sentence_1'].notna() & (sample_df['next_sentence_1'] != "")
    has_next_2 = sample_df['next_sentence_2'].notna() & (sample_df['next_sentence_2'] != "")
    
    has_full_window = has_prev_1 & has_prev_2 & has_next_1 & has_next_2
    has_any_prev = has_prev_1 | has_prev_2
    has_any_next = has_next_1 | has_next_2
    
    print(f"\n   Context Window Statistics (Window Size = 5):")
    print(f"      Sentences with full window (2+1+2): {has_full_window.sum():,} ({has_full_window.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with any previous context: {has_any_prev.sum():,} ({has_any_prev.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with any next context: {has_any_next.sum():,} ({has_any_next.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with prev-1 (i-2): {has_prev_1.sum():,} ({has_prev_1.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with prev-2 (i-1): {has_prev_2.sum():,} ({has_prev_2.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with next-1 (i+1): {has_next_1.sum():,} ({has_next_1.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with next-2 (i+2): {has_next_2.sum():,} ({has_next_2.sum()/len(sample_df)*100:.1f}%)")
    
    # Display DataFrame info
    print(f"\n\n📊 DataFrame Info:")
    print("-" * 100)
    sample_df.info()
    
    # Display full sample
    print(f"\n\n📋 Full Sample (first 10 rows):")
    print("=" * 100)
    display(sample_df.head(10))
    
else:
    print("\n⚠️  No output files found. Please run the processing cell first.")

print("\n" + "=" * 100)
print("✅ VERIFICATION COMPLETE")
print("=" * 100)

STAGE 1: VERIFICATION - SAMPLE OUTPUT

✅ Found 113 output files

📊 Sample Data from: Data_s1_1.csv
   Total sentences in file: 14,704
   Columns: ['sentence_id', 'article_id', 'date', 'source', 'previous_sentence_1', 'previous_sentence_2', 'main_sentence', 'next_sentence_1', 'next_sentence_2']

📋 First 5 Sentences (showing window structure):

🔹 Sentence 1:
   Sentence ID: f1_a1_s1
   Article ID: f1_a1
   Date: 2011-08-24 17:54:07
   Source: CNN_Articels_clean.csv
   Previous-1 (i-2): ""
   Previous-2 (i-1): ""
   Main (i): "Story highlightsBruno Senna will replace Nick Heidfeld for R..."
   Next-1 (i+1): ""
   Next-2 (i+2): ""

🔹 Sentence 2:
   Sentence ID: f1_a2_s1
   Article ID: f1_a2
   Date: 2011-09-02 11:18:24
   Source: CNN_Articels_clean.csv
   Previous-1 (i-2): ""
   Previous-2 (i-1): ""
   Main (i): "Story highlightsApp will make closed-off areas of Moorish pa..."
   Next-1 (i+1): "Entitled "The Hidden Alhambra," the project aims t..."
   Next-2 (i+2): "It will allow visitors 

,sentence_id,article_id,date,source,previous_sentence_1,previous_sentence_2,main_sentence,next_sentence_1,next_sentence_2
0,f1_a1_s1,f1_a1,2011-08-24 17:54:07,CNN_Articels_clean.csv,NaN,NaN,Story highlightsBruno Senna will replace Nick ...,NaN,NaN
1,f1_a2_s1,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,NaN,NaN,Story highlightsApp will make closed-off areas...,"Entitled ""The Hidden Alhambra,"" the project ai...",It will allow visitors to virtually explore de...
2,f1_a2_s2,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,NaN,Story highlightsApp will make closed-off areas...,"Entitled ""The Hidden Alhambra,"" the project ai...",It will allow visitors to virtually explore de...,"""It is a site that gets many visitors, we can ..."
3,f1_a2_s3,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,Story highlightsApp will make closed-off areas...,"Entitled ""The Hidden Alhambra,"" the project ai...",It will allow visitors to virtually explore de...,"""It is a site that gets many visitors, we can ...","""For many years now, we have worked with a mod..."
4,f1_a2_s4,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,"Entitled ""The Hidden Alhambra,"" the project ai...",It will allow visitors to virtually explore de...,"""It is a site that gets many visitors, we can ...","""For many years now, we have worked with a mod...","""Hidden areas that the application will bring ..."
5,f1_a2_s5,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,It will allow visitors to virtually explore de...,"""It is a site that gets many visitors, we can ...","""For many years now, we have worked with a mod...","""Hidden areas that the application will bring ...","""(Visitors will) be able to see images and vid..."
6,f1_a2_s6,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,"""It is a site that gets many visitors, we can ...","""For many years now, we have worked with a mod...","""Hidden areas that the application will bring ...","""(Visitors will) be able to see images and vid...","""They are not able to walk down the stairs int..."
7,f1_a2_s7,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,"""For many years now, we have worked with a mod...","""Hidden areas that the application will bring ...","""(Visitors will) be able to see images and vid...","""They are not able to walk down the stairs int...",The first phase of information-gathering for t...
8,f1_a2_s8,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,"""Hidden areas that the application will bring ...","""(Visitors will) be able to see images and vid...","""They are not able to walk down the stairs int...",The first phase of information-gathering for t...,They are aiming to have the app ready for the ...
9,f1_a2_s9,f1_a2,2011-09-02 11:18:24,CNN_Articels_clean.csv,"""(Visitors will) be able to see images and vid...","""They are not able to walk down the stairs int...",The first phase of information-gathering for t...,They are aiming to have the app ready for the ...,"""Once the technology began to improve, then we..."



✅ VERIFICATION COMPLETE


---

## Stage 2: Context-Aware Sentence Embedding

**Problem:** Isolated sentences can be semantically ambiguous. For example, a sentence about energy consumption may appear climate-related unless contextualized.

**Solution:** Each sentence is embedded together with its immediate neighbors using Sentence-BERT (SBERT) with two different window sizes.

**Two Window Sizes:**
1. **Window 3 (w3):** 1 previous + main + 1 next sentence
2. **Window 5 (w5):** 2 previous + main + 2 next sentences

**Method:** For a sentence `si`, we construct two contextual inputs:
- **w3:** `(si-1, si, si+1)` - Immediate context
- **w5:** `(si-2, si-1, si, si+1, si+2)` - Wider context

**Example (for sentence s2):**
- **w3:** `("AI is rapidly expanding.", "Data centers consume electricity.", "Governments regulate AI.")`
- **w5:** `("", "AI is rapidly expanding.", "Data centers consume electricity.", "Governments regulate AI.", "")`

**Input:** Stage 1 output (Data_s1_X.csv) with columns:
- sentence_id, article_id, date, source, previous_sentence_1, previous_sentence_2, main_sentence, next_sentence_1, next_sentence_2

**Output Format:**
- All Stage 1 columns PLUS:
- `w3_embedding`: 768-dimensional SBERT embedding with window size 3
- `w5_embedding`: 768-dimensional SBERT embedding with window size 5

**Processing Strategy:**
- Multi-threaded file processing (2 threads for CPU)
- Use SBERT model: `all-mpnet-base-v2` (768-dim, high quality)
- Each contextual input encoded as: `[prev] [SEP] [main] [SEP] [next]`
- Output saved to `Processed_Data/Stage_2/Data_s2_X.csv`

**Why Two Embeddings:**
- w3: Captures immediate local context (faster, more focused)
- w5: Captures broader narrative context (richer, more comprehensive)
- Allows comparison of different context window effects on shift detection

---

In [1]:
# ============================================================================
# STAGE 2: IMPORT ALL REQUIRED LIBRARIES
# ============================================================================
# Run this cell first if you want to run Stage 2 independently
# ============================================================================

print("=" * 100)
print("STAGE 2: IMPORTING REQUIRED LIBRARIES")
print("=" * 100)

# Core libraries from Stage 0
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
import json
import csv
import glob
from IPython.display import display, clear_output
from tqdm import tqdm

# Threading libraries
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Stage 2 specific libraries
from sentence_transformers import SentenceTransformer

print("✅ All libraries imported successfully")

# Define paths if not already defined
if 'STAGE_1_OUTPUT_FOLDER' not in dir():
    STAGE_1_OUTPUT_FOLDER = Path('Processed_Data/Stage_1')
    print(f"✅ STAGE_1_OUTPUT_FOLDER defined: {STAGE_1_OUTPUT_FOLDER.absolute()}")

if 'STAGE_2_OUTPUT_FOLDER' not in dir():
    STAGE_2_OUTPUT_FOLDER = Path('Processed_Data/Stage_2')
    print(f"✅ STAGE_2_OUTPUT_FOLDER defined: {STAGE_2_OUTPUT_FOLDER.absolute()}")

# Load SBERT model if not already loaded
if 'sbert_model' not in dir():
    print("\n🤖 Loading SBERT model...")
    print("   Device: CPU (limited to 2 cores)")
    
    # Limit to 2 CPU cores
    import os
    os.environ["OMP_NUM_THREADS"] = "2"
    os.environ["MKL_NUM_THREADS"] = "2"
    os.environ["OPENBLAS_NUM_THREADS"] = "2"
    os.environ["VECLIB_MAXIMUM_THREADS"] = "2"
    os.environ["NUMEXPR_NUM_THREADS"] = "2"
    
    sbert_model = SentenceTransformer('all-mpnet-base-v2', device='cpu')
    USE_GPU = False
    BATCH_SIZE = 64
    
    print(f"✅ SBERT model loaded on CPU (2 cores only)")
    print(f"   Embedding dim: {sbert_model.get_sentence_embedding_dimension()}")

# Define Stage 2 config if not already defined
if 'STAGE_2_CONFIG' not in dir():
    STAGE_2_CONFIG = {
        'input_folder': str(STAGE_1_OUTPUT_FOLDER),
        'output_folder': str(STAGE_2_OUTPUT_FOLDER),
        'output_file_prefix': 'Data_s2_',
        'model_name': 'all-mpnet-base-v2',
        'embedding_dim': sbert_model.get_sentence_embedding_dimension(),
        'use_gpu': False,
        'num_threads': 2,  # Only 2 threads for background processing
        'batch_size': 64
    }
    print(f"✅ STAGE_2_CONFIG defined (2 cores only)")

print("\n" + "=" * 100)
print("✅ STAGE 2 LIBRARIES READY - You can now run Stage 2 cells")
print("=" * 100)

STAGE 2: IMPORTING REQUIRED LIBRARIES
✅ All libraries imported successfully
✅ STAGE_1_OUTPUT_FOLDER defined: /home/hp/INLP/Naretve_Shift/Main_Code/Processed_Data/Stage_1
✅ STAGE_2_OUTPUT_FOLDER defined: /home/hp/INLP/Naretve_Shift/Main_Code/Processed_Data/Stage_2

🤖 Loading SBERT model...
   Device: CPU (limited to 2 cores)
✅ SBERT model loaded on CPU (2 cores only)
   Embedding dim: 768
✅ STAGE_2_CONFIG defined (2 cores only)

✅ STAGE 2 LIBRARIES READY - You can now run Stage 2 cells


In [2]:
# ============================================================================
# STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - SETUP
# ============================================================================
# Install and import Sentence-BERT for semantic embeddings
# ============================================================================

print("=" * 100)
print("STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - SETUP")
print("=" * 100)

# Import sentence transformers library
try:
    from sentence_transformers import SentenceTransformer
    print("✅ sentence-transformers already installed")
except ImportError:
    print("📥 Installing sentence-transformers...")
    import subprocess
    import sys
    import pathlib
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
    from sentence_transformers import SentenceTransformer
    print("✅ sentence-transformers installed successfully")

print("\n✅ All libraries for Stage 2 ready")

# ----------------------------------------------------------------------------
# Load SBERT Model (CPU Mode)
# ----------------------------------------------------------------------------
print("\n🤖 Loading Sentence-BERT Model...")
print("   Model: all-mpnet-base-v2 (768-dimensional embeddings)")
print("   This may take a moment on first run (downloading model ~420MB)...")
print("   Device: CPU (forced for stability)")

# Force CPU usage
sbert_model = SentenceTransformer('all-mpnet-base-v2', device='cpu')

# Limit to 2 CPU cores for background processing
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["VECLIB_MAXIMUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

print("✅ SBERT model loaded successfully on CPU")
print(f"   CPU cores limited to: 2 (for background processing)")
print(f"   Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")

# Set configuration for CPU
USE_GPU = False
BATCH_SIZE = 64  # Optimal batch size for CPU

# ----------------------------------------------------------------------------
# Configure Output Folder
# ----------------------------------------------------------------------------
print("\n📁 Configuring Output Folder...")

# Create output folder for Stage 2
STAGE_2_OUTPUT_FOLDER = Path('Processed_Data/Stage_2')
STAGE_2_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"✅ Output folder: {STAGE_2_OUTPUT_FOLDER.absolute()}")

# Configuration for Stage 2
STAGE_2_CONFIG = {
    'input_folder': str(STAGE_1_OUTPUT_FOLDER),
    'output_folder': str(STAGE_2_OUTPUT_FOLDER),
    'output_file_prefix': 'Data_s2_',
    'model_name': 'all-mpnet-base-v2',
    'embedding_dim': sbert_model.get_sentence_embedding_dimension(),
    'device': 'cpu',
    'use_gpu': False,
    'num_threads': 2,  # Only 2 threads to leave CPU available for other work
    'batch_size': 64   # Optimal batch size for CPU
}

print(f"\n⚙️  Stage 2 Configuration:")
for key, value in STAGE_2_CONFIG.items():
    print(f"   {key}: {value}")

print(f"\n💡 Background Processing Mode:")
print(f"   - CPU cores: Limited to 2 cores")
print(f"   - Threads: 2 (leaves CPU available for other tasks)")
print(f"   - Batch size: 64")
print(f"   - Expected time: ~80-100s per file (~8-10 hours total)")
print(f"   - ✅ You can work on other tasks while this runs!")

print("\n" + "=" * 100)
print("✅ STAGE 2 SETUP COMPLETE")
print("=" * 100)

STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - SETUP
✅ sentence-transformers already installed

✅ All libraries for Stage 2 ready

🤖 Loading Sentence-BERT Model...
   Model: all-mpnet-base-v2 (768-dimensional embeddings)
   This may take a moment on first run (downloading model ~420MB)...
   Device: CPU (forced for stability)
✅ SBERT model loaded successfully on CPU
   CPU cores limited to: 2 (for background processing)
   Embedding dimension: 768

📁 Configuring Output Folder...
✅ Output folder: /home/hp/INLP/Naretve_Shift/Main_Code/Processed_Data/Stage_2

⚙️  Stage 2 Configuration:
   input_folder: Processed_Data/Stage_1
   output_folder: Processed_Data/Stage_2
   output_file_prefix: Data_s2_
   model_name: all-mpnet-base-v2
   embedding_dim: 768
   device: cpu
   use_gpu: False
   num_threads: 2
   batch_size: 64

💡 Background Processing Mode:
   - CPU cores: Limited to 2 cores
   - Threads: 2 (leaves CPU available for other tasks)
   - Batch size: 64
   - Expected time: ~80-100s per fi

In [ ]:
# ============================================================================
# STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - PROCESSING FUNCTIONS
# ============================================================================
# Define functions for creating context-aware embeddings with two window sizes
# ============================================================================

def create_contextual_input_w3(prev_sent_i_minus_1, main_sent, next_sent_i_plus_1):
    """
    Create contextual input with window size 3 (1 previous + main + 1 next).
    
    Window 3 uses positions: (i-1, i, i+1)
    - prev_sent_i_minus_1: sentence at position (i-1) = previous_sentence_2 in Stage 1
    - main_sent: sentence at position (i) = main_sentence in Stage 1
    - next_sent_i_plus_1: sentence at position (i+1) = next_sentence_1 in Stage 1
    
    Args:
        prev_sent_i_minus_1 (str): Previous sentence 1 position back (i-1)
        main_sent (str): Main sentence (current, position i)
        next_sent_i_plus_1 (str): Next sentence 1 position ahead (i+1)
    
    Returns:
        str: Concatenated contextual input with [SEP] tokens (window size 3)
    """
    parts = []
    
    # Add previous sentence (i-1) if available
    if prev_sent_i_minus_1 and str(prev_sent_i_minus_1).strip():
        parts.append(str(prev_sent_i_minus_1).strip())
    
    # Always add main sentence (i)
    parts.append(str(main_sent).strip())
    
    # Add next sentence (i+1) if available
    if next_sent_i_plus_1 and str(next_sent_i_plus_1).strip():
        parts.append(str(next_sent_i_plus_1).strip())
    
    return " [SEP] ".join(parts)


def create_contextual_input_w5(prev_sent_i_minus_2, prev_sent_i_minus_1, main_sent, 
                                next_sent_i_plus_1, next_sent_i_plus_2):
    """
    Create contextual input with window size 5 (2 previous + main + 2 next).
    
    Window 5 uses positions: (i-2, i-1, i, i+1, i+2)
    - prev_sent_i_minus_2: sentence at position (i-2) = previous_sentence_1 in Stage 1
    - prev_sent_i_minus_1: sentence at position (i-1) = previous_sentence_2 in Stage 1
    - main_sent: sentence at position (i) = main_sentence in Stage 1
    - next_sent_i_plus_1: sentence at position (i+1) = next_sentence_1 in Stage 1
    - next_sent_i_plus_2: sentence at position (i+2) = next_sentence_2 in Stage 1
    
    Args:
        prev_sent_i_minus_2 (str): Previous sentence 2 positions back (i-2)
        prev_sent_i_minus_1 (str): Previous sentence 1 position back (i-1)
        main_sent (str): Main sentence (current, position i)
        next_sent_i_plus_1 (str): Next sentence 1 position ahead (i+1)
        next_sent_i_plus_2 (str): Next sentence 2 positions ahead (i+2)
    
    Returns:
        str: Concatenated contextual input with [SEP] tokens (window size 5)
    """
    parts = []
    
    # Add previous sentence (i-2) if available
    if prev_sent_i_minus_2 and str(prev_sent_i_minus_2).strip():
        parts.append(str(prev_sent_i_minus_2).strip())
    
    # Add previous sentence (i-1) if available
    if prev_sent_i_minus_1 and str(prev_sent_i_minus_1).strip():
        parts.append(str(prev_sent_i_minus_1).strip())
    
    # Always add main sentence (i)
    parts.append(str(main_sent).strip())
    
    # Add next sentence (i+1) if available
    if next_sent_i_plus_1 and str(next_sent_i_plus_1).strip():
        parts.append(str(next_sent_i_plus_1).strip())
    
    # Add next sentence (i+2) if available
    if next_sent_i_plus_2 and str(next_sent_i_plus_2).strip():
        parts.append(str(next_sent_i_plus_2).strip())
    
    return " [SEP] ".join(parts)


def process_single_file_stage2(input_file_path, file_number, total_files):
    """
    Process a single Stage 1 file and create context-aware embeddings with two window sizes.
    
    Args:
        input_file_path (Path): Path to the Stage 1 CSV file
        file_number (int): File number (for tracking)
        total_files (int): Total number of files
    
    Returns:
        tuple: (file_number, sentence_count, output_file_path, success, elapsed_time)
    """
    try:
        start_time = time.time()
        
        # Read the Stage 1 CSV file
        df = pd.read_csv(input_file_path)
        
        # Validate required columns (updated for window size 5 structure)
        required_cols = ['sentence_id', 'article_id', 'date', 'source', 
                        'previous_sentence_1', 'previous_sentence_2', 'main_sentence', 
                        'next_sentence_1', 'next_sentence_2']
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️  Skipping {input_file_path.name}: Missing required columns")
            return (file_number, 0, None, False, 0)
        
        # Replace NaN with empty strings
        df = df.fillna("")
        
        # Create contextual inputs for both window sizes
        contextual_inputs_w3 = []
        contextual_inputs_w5 = []
        
        for idx, row in df.iterrows():
            # Window 3: Use positions (i-1, i, i+1)
            # - i-1 = previous_sentence_2
            # - i = main_sentence
            # - i+1 = next_sentence_1
            input_w3 = create_contextual_input_w3(
                row['previous_sentence_2'],  # i-1 (prev_sent_i_minus_1)
                row['main_sentence'],         # i (main_sent)
                row['next_sentence_1']        # i+1 (next_sent_i_plus_1)
            )
            contextual_inputs_w3.append(input_w3)
            
            # Window 5: Use positions (i-2, i-1, i, i+1, i+2)
            # - i-2 = previous_sentence_1
            # - i-1 = previous_sentence_2
            # - i = main_sentence
            # - i+1 = next_sentence_1
            # - i+2 = next_sentence_2
            input_w5 = create_contextual_input_w5(
                row['previous_sentence_1'],  # i-2 (prev_sent_i_minus_2)
                row['previous_sentence_2'],  # i-1 (prev_sent_i_minus_1)
                row['main_sentence'],         # i (main_sent)
                row['next_sentence_1'],       # i+1 (next_sent_i_plus_1)
                row['next_sentence_2']        # i+2 (next_sent_i_plus_2)
            )
            contextual_inputs_w5.append(input_w5)
        
        # Generate embeddings in batches for efficiency
        batch_size = STAGE_2_CONFIG['batch_size']
        
        # Generate w3 embeddings
        all_embeddings_w3 = []
        for i in range(0, len(contextual_inputs_w3), batch_size):
            batch = contextual_inputs_w3[i:i + batch_size]
            batch_embeddings = sbert_model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
            all_embeddings_w3.extend(batch_embeddings)
        
        # Generate w5 embeddings
        all_embeddings_w5 = []
        for i in range(0, len(contextual_inputs_w5), batch_size):
            batch = contextual_inputs_w5[i:i + batch_size]
            batch_embeddings = sbert_model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
            all_embeddings_w5.extend(batch_embeddings)
        
        # Convert embeddings lists to numpy arrays
        embeddings_array_w3 = np.array(all_embeddings_w3)
        embeddings_array_w5 = np.array(all_embeddings_w5)
        
        # Add embeddings as new columns (store as comma-separated strings for CSV)
        df['w3_embedding'] = [','.join(map(str, emb)) for emb in embeddings_array_w3]
        df['w5_embedding'] = [','.join(map(str, emb)) for emb in embeddings_array_w5]
        
        # Define output file path
        output_file = STAGE_2_OUTPUT_FOLDER / f"Data_s2_{file_number}.csv"
        
        # Save to CSV
        df.to_csv(output_file, index=False)
        
        elapsed_time = time.time() - start_time
        
        return (file_number, len(df), output_file, True, elapsed_time)
        
    except Exception as e:
        print(f"\n❌ Error processing {input_file_path.name}: {e}")
        import traceback
        traceback.print_exc()
        return (file_number, 0, None, False, 0)


print("✅ Stage 2 processing functions defined:")
print("   - create_contextual_input_w3(): Window size 3 (1 prev + main + 1 next)")
print("   - create_contextual_input_w5(): Window size 5 (2 prev + main + 2 next)")
print("   - process_single_file_stage2(): Generate both w3 and w5 embeddings for file")

✅ Stage 2 processing functions defined:
   - create_contextual_input(): Combine sentences with [SEP]
   - process_single_file_stage2(): Generate embeddings for file


In [ ]:
# ============================================================================
# STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - MAIN PROCESSING
# ============================================================================
# Process all Stage 1 files with multi-threading
# ============================================================================

print("=" * 100)
print("STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - MAIN PROCESSING")
print("=" * 100)

# Get all Stage 1 output files
stage1_files = sorted(STAGE_1_OUTPUT_FOLDER.glob('Data_s1_*.csv'))
total_files = len(stage1_files)
num_threads = STAGE_2_CONFIG['num_threads']

print(f"\n📊 Processing Overview:")
print(f"   Total files to process: {total_files:,}")
print(f"   Input folder: {STAGE_1_OUTPUT_FOLDER.absolute()}")
print(f"   Output folder: {STAGE_2_OUTPUT_FOLDER.absolute()}")
print(f"   Number of threads: {num_threads}")
print(f"   Batch size: {STAGE_2_CONFIG['batch_size']}")
print(f"   Embedding dimension: {STAGE_2_CONFIG['embedding_dim']}")
print(f"   Window sizes: w3 (1+1+1) and w5 (2+1+2)")
print(f"   Output columns: w3_embedding, w5_embedding")

# Initialize tracking variables
processed_files = 0
total_sentences = 0
successful_files = []
failed_files = []
start_time_overall = time.time()

# Thread tracking
thread_status = {}
completed_files = []

# Thread-safe lock
progress_lock = threading.Lock()

def mark_thread_start_s2(thread_id, file_number, file_name):
    """Mark that a thread has started processing a file"""
    with progress_lock:
        thread_status[thread_id] = {
            'file_number': file_number,
            'file_name': file_name,
            'start_time': time.time()
        }

def mark_thread_complete_s2(thread_id):
    """Mark that a thread has completed processing"""
    with progress_lock:
        if thread_id in thread_status:
            del thread_status[thread_id]

def update_progress_s2(file_number, file_name, sentence_count, output_file, success, elapsed):
    """Thread-safe progress update"""
    global processed_files, total_sentences
    
    with progress_lock:
        if success:
            processed_files += 1
            total_sentences += sentence_count
            file_info = {
                'file_number': file_number,
                'file_name': file_name,
                'sentences': sentence_count,
                'output': output_file.name,
                'time': elapsed
            }
            successful_files.append(file_info)
            completed_files.append(file_info)
            
            if len(completed_files) > 10:
                completed_files.pop(0)
            
            avg_time_per_file = (time.time() - start_time_overall) / processed_files
            estimated_remaining_time = avg_time_per_file * (total_files - processed_files)
            
            # Clear and display progress
            clear_output(wait=True)
            print("=" * 120)
            print(f"🚀 STAGE 2 PROCESSING IN PROGRESS - {num_threads} THREADS ACTIVE")
            print("=" * 120)
            
            # Overall Progress bar
            progress_pct = (processed_files / total_files) * 100
            bar_length = 60
            filled_length = int(bar_length * processed_files / total_files)
            bar = '█' * filled_length + '░' * (bar_length - filled_length)
            
            print(f"\n📊 Overall Progress:")
            print(f"   [{bar}] {progress_pct:.1f}%")
            print(f"   Files: {processed_files}/{total_files} | Sentences: {total_sentences:,}")
            print(f"   Time Elapsed: {(time.time() - start_time_overall)/60:.1f} min | ETA: {estimated_remaining_time/60:.1f} min")
            print(f"   Avg Speed: {avg_time_per_file:.2f}s/file | Processing Rate: {processed_files/(time.time() - start_time_overall)*60:.1f} files/min")
            
            # Thread-by-thread status
            print(f"\n🔄 Active Threads ({len(thread_status)}/{num_threads}):")
            print("-" * 120)
            if thread_status:
                sorted_threads = sorted(thread_status.items(), key=lambda x: x[0])
                for thread_id, status in sorted_threads:
                    elapsed_thread = time.time() - status['start_time']
                    thread_bar_length = 15
                    if processed_files > 0:
                        thread_progress = min(1.0, elapsed_thread / avg_time_per_file)
                    else:
                        thread_progress = 0.3
                    thread_filled = int(thread_bar_length * thread_progress)
                    thread_bar = '█' * thread_filled + '░' * (thread_bar_length - thread_filled)
                    
                    print(f"   Thread {thread_id:2d}: [{thread_bar}] Processing File {status['file_number']:3d} "
                          f"({status['file_name']:<25}) - {elapsed_thread:.1f}s")
            else:
                print("   All threads idle")
            
            # Show recently completed files
            print(f"\n✅ Recently Completed Files:")
            print("-" * 120)
            for file_info in completed_files[-8:]:
                print(f"   File {file_info['file_number']:3d}: {file_info['output']:<25} | "
                      f"Sentences: {file_info['sentences']:>7,} | Time: {file_info['time']:>5.2f}s")
            
        else:
            failed_files.append({
                'file_number': file_number,
                'file_name': file_name
            })
            print(f"\n❌ Failed: File {file_number} - {file_name}")

print(f"\n🚀 Starting multi-threaded embedding generation...")
print("=" * 120)

# Initial progress display
clear_output(wait=True)
print("=" * 120)
print(f"🚀 STAGE 2 STARTING - {num_threads} THREADS INITIALIZING")
print("=" * 120)
print(f"\n📊 Ready to process {total_files:,} files")
print(f"   Creating {STAGE_2_CONFIG['embedding_dim']}-dimensional embeddings...")

# Wrapper function to track thread activity
def process_file_with_tracking_s2(input_file_path, file_idx, total_files, thread_id):
    """Wrapper to track thread activity during processing"""
    mark_thread_start_s2(thread_id, file_idx, input_file_path.name)
    
    try:
        result = process_single_file_stage2(input_file_path, file_idx, total_files)
        return result
    finally:
        mark_thread_complete_s2(thread_id)

# Process files using ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    futures = []
    future_to_info = {}
    
    for file_idx, input_file_path in enumerate(stage1_files, 1):
        thread_id = (file_idx - 1) % num_threads
        future = executor.submit(process_file_with_tracking_s2, input_file_path, file_idx, total_files, thread_id)
        futures.append(future)
        future_to_info[future] = (file_idx, input_file_path, thread_id)
    
    # Process completed tasks as they finish
    for future in as_completed(futures):
        file_idx, input_file_path, thread_id = future_to_info[future]
        try:
            result = future.result()
            file_number, sentence_count, output_file, success, elapsed = result
            update_progress_s2(file_number, input_file_path.name, sentence_count, output_file, success, elapsed)
        except Exception as e:
            mark_thread_complete_s2(thread_id)
            print(f"\n❌ Exception processing file {file_idx}: {e}")
            update_progress_s2(file_idx, input_file_path.name, 0, None, False, 0)

# Calculate final statistics
elapsed_time_overall = time.time() - start_time_overall

# Clear and show final summary
clear_output(wait=True)

print("\n" + "=" * 100)
print("📈 STAGE 2 PROCESSING COMPLETE")
print("=" * 100)

print(f"\n✅ Summary:")
print(f"   Files processed successfully: {len(successful_files):,}/{total_files:,}")
print(f"   Files failed: {len(failed_files):,}")
print(f"   Total sentences embedded: {total_sentences:,}")
print(f"   Embedding dimension: {STAGE_2_CONFIG['embedding_dim']}")
print(f"   Total processing time: {elapsed_time_overall/60:.2f} minutes")
if len(successful_files) > 0:
    print(f"   Average time per file: {elapsed_time_overall/len(successful_files):.2f} seconds")
    print(f"   Average sentences per file: {total_sentences/len(successful_files):.0f}")
print(f"   Processing speed: {len(successful_files)/(elapsed_time_overall/60):.1f} files/minute")

# Show sample of successful files
if successful_files:
    print(f"\n📋 Sample of Processed Files (first 10):")
    print("-" * 100)
    print(f"{'File #':<10} {'Sentences':<12} {'Output File':<30} {'Time (s)':<10}")
    print("-" * 100)
    sorted_files = sorted(successful_files, key=lambda x: x['file_number'])
    for file_info in sorted_files[:10]:
        print(f"{file_info['file_number']:<10} {file_info['sentences']:<12,} "
              f"{file_info['output']:<30} {file_info['time']:<10.2f}")
    
    if len(successful_files) > 10:
        print(f"\n   ... ({len(successful_files) - 10} more files)")

# Show failed files if any
if failed_files:
    print(f"\n⚠️  Failed Files ({len(failed_files)}):")
    for file_info in sorted(failed_files, key=lambda x: x['file_number']):
        print(f"   {file_info['file_number']}. {file_info['file_name']}")

# Save processing summary
summary_file = STAGE_2_OUTPUT_FOLDER / 'processing_summary.json'
summary_data = {
    'total_files': total_files,
    'successful_files': len(successful_files),
    'failed_files': len(failed_files),
    'total_sentences': total_sentences,
    'embedding_dimension': STAGE_2_CONFIG['embedding_dim'],
    'model_name': STAGE_2_CONFIG['model_name'],
    'processing_time_minutes': elapsed_time_overall / 60,
    'average_time_per_file_seconds': elapsed_time_overall / len(successful_files) if len(successful_files) > 0 else 0,
    'processing_speed_files_per_minute': len(successful_files) / (elapsed_time_overall / 60) if elapsed_time_overall > 0 else 0,
    'num_threads': num_threads,
    'batch_size': STAGE_2_CONFIG['batch_size'],
    'output_folder': str(STAGE_2_OUTPUT_FOLDER),
    'timestamp': datetime.now().isoformat()
}

with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\n💾 Processing summary saved: {summary_file.name}")

print("\n" + "=" * 100)
print("🎉 STAGE 2 COMPLETE!")
print("=" * 100)
print(f"\n💡 Next Stage: Topic Embedding Construction (Stage 3)")
print(f"   Input: {len(successful_files):,} files with {total_sentences:,} embedded sentences")
print(f"   Location: {STAGE_2_OUTPUT_FOLDER.absolute()}")
print("=" * 100)

🚀 STAGE 2 PROCESSING IN PROGRESS - 2 THREADS ACTIVE

📊 Overall Progress:
   [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 3.2%
   Files: 11/347 | Sentences: 48,835
   Time Elapsed: 425.5 min | ETA: 12997.0 min
   Avg Speed: 2320.89s/file | Processing Rate: 0.0 files/min

🔄 Active Threads (0/2):
------------------------------------------------------------------------------------------------------------------------
   All threads idle

✅ Recently Completed Files:
------------------------------------------------------------------------------------------------------------------------
   File   5: Data_s2_5.csv             | Sentences:   2,084 | Time: 343.71s
   File   6: Data_s2_6.csv             | Sentences:   2,059 | Time: 387.07s
   File   7: Data_s2_7.csv             | Sentences:   2,082 | Time: 367.40s
   File   8: Data_s2_8.csv             | Sentences:   2,242 | Time: 390.44s
   File   9: Data_s2_9.csv             | Sentences:   2,109 | Time: 384.66s
   File  10: Dat

In [ ]:
# ============================================================================
# STAGE 2: VERIFICATION AND SAMPLE DISPLAY
# ============================================================================
# Verify output and display sample embeddings (both w3 and w5)
# ============================================================================

print("=" * 100)
print("STAGE 2: VERIFICATION - SAMPLE OUTPUT")
print("=" * 100)

# Get list of output files
output_files_s2 = sorted(STAGE_2_OUTPUT_FOLDER.glob('Data_s2_*.csv'))

if len(output_files_s2) > 0:
    print(f"\n✅ Found {len(output_files_s2):,} output files")
    
    # Load first file as sample
    sample_file_s2 = output_files_s2[0]
    print(f"\n📊 Loading sample file: {sample_file_s2.name}")
    print("   (This may take a moment due to large embedding columns...)")
    
    # Read only first few rows for quick verification
    sample_df_s2 = pd.read_csv(sample_file_s2, nrows=100)
    
    print(f"\n📊 Sample Data from: {sample_file_s2.name}")
    print(f"   Total sentences (in sample): {len(sample_df_s2):,}")
    print(f"   Columns: {list(sample_df_s2.columns)}")
    
    # Display first 3 sentences with context
    print(f"\n📋 First 3 Sentences (showing structure with both embeddings):")
    print("=" * 100)
    
    for idx in range(min(3, len(sample_df_s2))):
        row = sample_df_s2.iloc[idx]
        
        prev_1 = str(row['previous_sentence_1']) if pd.notna(row['previous_sentence_1']) else ""
        prev_2 = str(row['previous_sentence_2']) if pd.notna(row['previous_sentence_2']) else ""
        main_sent = str(row['main_sentence']) if pd.notna(row['main_sentence']) else ""
        next_1 = str(row['next_sentence_1']) if pd.notna(row['next_sentence_1']) else ""
        next_2 = str(row['next_sentence_2']) if pd.notna(row['next_sentence_2']) else ""
        
        # Parse embeddings (first few values)
        w3_embedding_str = str(row['w3_embedding'])
        w3_embedding_values = w3_embedding_str.split(',')[:5]  # Show first 5 values
        
        w5_embedding_str = str(row['w5_embedding'])
        w5_embedding_values = w5_embedding_str.split(',')[:5]  # Show first 5 values
        
        print(f"\n🔹 Sentence {idx + 1}:")
        print(f"   Sentence ID: {row['sentence_id']}")
        print(f"   Article ID: {row['article_id']}")
        print(f"   Date: {row['date']}")
        print(f"   Source: {row['source']}")
        print(f"   Context Window (5 sentences):")
        print(f"      Previous-1 (i-2): \"{prev_1[:40]}{'...' if len(prev_1) > 40 else ''}\"")
        print(f"      Previous-2 (i-1): \"{prev_2[:40]}{'...' if len(prev_2) > 40 else ''}\"")
        print(f"      Main (i):         \"{main_sent[:50]}{'...' if len(main_sent) > 50 else ''}\"")
        print(f"      Next-1 (i+1):     \"{next_1[:40]}{'...' if len(next_1) > 40 else ''}\"")
        print(f"      Next-2 (i+2):     \"{next_2[:40]}{'...' if len(next_2) > 40 else ''}\"")
        print(f"   w3_embedding (window 3): [{', '.join(w3_embedding_values[:5])}...] ({len(w3_embedding_values)} dims)")
        print(f"   w5_embedding (window 5): [{', '.join(w5_embedding_values[:5])}...] ({len(w5_embedding_values)} dims)")
    
    # Statistics
    print(f"\n\n📊 Statistics for Sample File:")
    print("-" * 100)
    
    # Count unique articles
    unique_articles = sample_df_s2['article_id'].nunique()
    avg_sentences_per_article = len(sample_df_s2) / unique_articles
    
    print(f"   Unique articles (in sample): {unique_articles:,}")
    print(f"   Average sentences per article: {avg_sentences_per_article:.1f}")
    
    # Verify embedding columns
    print(f"\n   Embedding Verification:")
    
    # Check w3 embedding
    sample_w3_embedding = sample_df_s2['w3_embedding'].iloc[0]
    w3_embedding_dim = len(str(sample_w3_embedding).split(','))
    print(f"      w3_embedding dimension: {w3_embedding_dim}")
    
    # Check w5 embedding
    sample_w5_embedding = sample_df_s2['w5_embedding'].iloc[0]
    w5_embedding_dim = len(str(sample_w5_embedding).split(','))
    print(f"      w5_embedding dimension: {w5_embedding_dim}")
    
    print(f"      Expected dimension: {STAGE_2_CONFIG['embedding_dim']}")
    
    if w3_embedding_dim == STAGE_2_CONFIG['embedding_dim'] and w5_embedding_dim == STAGE_2_CONFIG['embedding_dim']:
        print(f"      ✅ Both embedding dimensions match!")
    else:
        print(f"      ⚠️  Warning: Dimension mismatch!")
    
    # Count sentences with context
    has_prev_1 = sample_df_s2['previous_sentence_1'].notna() & (sample_df_s2['previous_sentence_1'] != "")
    has_prev_2 = sample_df_s2['previous_sentence_2'].notna() & (sample_df_s2['previous_sentence_2'] != "")
    has_next_1 = sample_df_s2['next_sentence_1'].notna() & (sample_df_s2['next_sentence_1'] != "")
    has_next_2 = sample_df_s2['next_sentence_2'].notna() & (sample_df_s2['next_sentence_2'] != "")
    
    has_full_w5 = has_prev_1 & has_prev_2 & has_next_1 & has_next_2
    has_full_w3 = has_prev_2 & has_next_1  # For w3, only need prev_2 (i-1) and next_1 (i+1)
    
    print(f"\n   Context Statistics:")
    print(f"      Sentences with full w5 context (2+1+2): {has_full_w5.sum():,} ({has_full_w5.sum()/len(sample_df_s2)*100:.1f}%)")
    print(f"      Sentences with full w3 context (1+1+1): {has_full_w3.sum():,} ({has_full_w3.sum()/len(sample_df_s2)*100:.1f}%)")
    
    # Display DataFrame info (excluding embedding columns for readability)
    print(f"\n\n📊 DataFrame Info (excluding embedding columns):")
    print("-" * 100)
    cols_to_show = [col for col in sample_df_s2.columns if col not in ['w3_embedding', 'w5_embedding']]
    sample_df_s2[cols_to_show].info()
    
    # Display sample without embedding columns
    print(f"\n\n📋 Sample Data (first 10 rows, excluding embedding columns):")
    print("=" * 100)
    display(sample_df_s2[cols_to_show].head(10))
    
    # Test: Parse both embeddings back to arrays
    print(f"\n\n🧪 Embedding Test:")
    print("-" * 100)
    
    # Test w3 embedding
    test_w3_str = sample_df_s2['w3_embedding'].iloc[0]
    test_w3_array = np.array([float(x) for x in test_w3_str.split(',')])
    print(f"   w3_embedding (Window 3):")
    print(f"      Successfully parsed to numpy array")
    print(f"      Shape: {test_w3_array.shape}")
    print(f"      First 10 values: {test_w3_array[:10]}")
    print(f"      Min: {test_w3_array.min():.4f}, Max: {test_w3_array.max():.4f}")
    print(f"      Mean: {test_w3_array.mean():.4f}, Std: {test_w3_array.std():.4f}")
    
    # Test w5 embedding
    test_w5_str = sample_df_s2['w5_embedding'].iloc[0]
    test_w5_array = np.array([float(x) for x in test_w5_str.split(',')])
    print(f"\n   w5_embedding (Window 5):")
    print(f"      Successfully parsed to numpy array")
    print(f"      Shape: {test_w5_array.shape}")
    print(f"      First 10 values: {test_w5_array[:10]}")
    print(f"      Min: {test_w5_array.min():.4f}, Max: {test_w5_array.max():.4f}")
    print(f"      Mean: {test_w5_array.mean():.4f}, Std: {test_w5_array.std():.4f}")
    
    # Compare embeddings
    print(f"\n   Comparison:")
    cosine_sim = np.dot(test_w3_array, test_w5_array) / (np.linalg.norm(test_w3_array) * np.linalg.norm(test_w5_array))
    print(f"      Cosine similarity between w3 and w5: {cosine_sim:.4f}")
    print(f"      (Higher values indicate similar embeddings despite different window sizes)")
    
else:
    print("\n⚠️  No output files found. Please run the processing cell first.")

print("\n" + "=" * 100)
print("✅ VERIFICATION COMPLETE")
print("=" * 100)